<a href="https://colab.research.google.com/github/karrosue-cpu/Data-Analytics/blob/main/Karrosue_Prince_Capstone_Project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import requests
from io import StringIO

url = "https://ohss.dhs.gov/sites/default/files/2025-05/state_data_2013-2023_20250514_3.csv"

headers = {
    "User-Agent": "Mozilla/5.0"
}

response = requests.get(url, headers=headers)

df = pd.read_csv(StringIO(response.text))

# PRINT DATASET
print("Dataset Loaded Successfully")
print(df.head())      # first 5 rows
print("\nColumns:\n", df.columns)

In [ ]:
# ============================================================
# U.S. Immigration Trends by State, 2013–2023
# Capstone Project — Karrosue Prince
# Dataset: DHS/OHSS State Immigration Data Flat File
# ============================================================

# STEP 1: Import libraries
import pandas as pd
import matplotlib.pyplot as plt
import plotly.express as px
import requests
from io import StringIO

pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:,.0f}'.format)

# STEP 2: Load the real DHS/OHSS dataset
url = "https://ohss.dhs.gov/sites/default/files/2025-05/state_data_2013-2023_20250514_3.csv"

headers = {"User-Agent": "Mozilla/5.0"}
response = requests.get(url, headers=headers)

if response.status_code == 200:
    df = pd.read_csv(StringIO(response.text))
    print("Dataset loaded successfully from DHS/OHSS URL.")
else:
    print("Direct download failed. Upload CSV manually to Colab, then use:")
    print("df = pd.read_csv('state_data_2013-2023_20250514_3.csv')")

# STEP 3: Preview the data
df.head()

In [ ]:
# STEP 4: Check exact column names
df.columns.tolist()

In [ ]:
# STEP 5: Verify required columns exist
required_columns = [
    'State',
    'Year',
    'Population',
    'Lawful Permanent Residents Total',
    'Adjustments Total',
    'New Arrivals Total',
    'Naturalizations Total',
    'Refugees Total',
    'Asylees Total'
]

missing_columns = [col for col in required_columns if col not in df.columns]

if missing_columns:
    print("Missing columns:", missing_columns)
else:
    print("All required columns are present.")

In [ ]:
# STEP 6: Clean numeric columns
numeric_columns = [
    'Year',
    'Population',
    'Lawful Permanent Residents Total',
    'Adjustments Total',
    'New Arrivals Total',
    'Naturalizations Total',
    'Refugees Total',
    'Asylees Total'
]

for col in numeric_columns:
    df[col] = (
        df[col]
        .astype(str)
        .str.replace(',', '', regex=False)
        .str.strip()
    )
    df[col] = pd.to_numeric(df[col], errors='coerce')

df_clean = df.dropna(subset=['State', 'Year']).copy()

category_columns = [
    'Lawful Permanent Residents Total',
    'Naturalizations Total',
    'Refugees Total',
    'Asylees Total'
]

df_clean[category_columns] = df_clean[category_columns].fillna(0)

df_clean.info()

In [ ]:
# STEP 7: Create total immigration activity column
# NOTE: Adjustments and New Arrivals are NOT added because they are part of LPR total.
# Adding them again could double-count.

df_clean['Total Immigration Activity'] = (
    df_clean['Lawful Permanent Residents Total'] +
    df_clean['Naturalizations Total'] +
    df_clean['Refugees Total'] +
    df_clean['Asylees Total']
)

df_clean[['State', 'Year', 'Total Immigration Activity']].head()

In [ ]:

# Which states had the highest lawful permanent residents from 2013–2023?

top_lpr_states = (
    df_clean.groupby('State')['Lawful Permanent Residents Total']
    .sum()
    .sort_values(ascending=False)
    .head(10)
)

top_lpr_states

In [ ]:
# Graph 1: Top 10 states by lawful permanent residents

plt.figure(figsize=(12, 6))
top_lpr_states.plot(kind='bar')
plt.title('Top 10 States by Lawful Permanent Residents, 2013–2023')
plt.xlabel('State')
plt.ylabel('Lawful Permanent Residents')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

In [ ]:

# Which states had the highest total immigration activity?

top_total_states = (
    df_clean.groupby('State')['Total Immigration Activity']
    .sum()
    .sort_values(ascending=False)
    .head(10)
)

top_total_states

In [ ]:
# Graph 2: Top 10 states by total immigration activity

plt.figure(figsize=(12, 6))
top_total_states.plot(kind='bar')
plt.title('Top 10 States by Total Immigration Activity, 2013–2023')
plt.xlabel('State')
plt.ylabel('Total Immigration Activity')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

In [ ]:
# QUESTION 3:
# Which immigration categories were the largest overall?

category_totals = df_clean[
    [
        'Lawful Permanent Residents Total',
        'Naturalizations Total',
        'Refugees Total',
        'Asylees Total'
    ]
].sum().sort_values(ascending=False)

category_totals

In [ ]:
# Graph 3: Immigration categories

plt.figure(figsize=(10, 6))
category_totals.plot(kind='bar')
plt.title('Total Immigration Activity by Category, 2013–2023')
plt.xlabel('Immigration Category')
plt.ylabel('Total Count')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

In [ ]:
# QUESTION 4:
# How did immigration activity change over time?

yearly_trend = (
    df_clean.groupby('Year')['Total Immigration Activity']
    .sum()
    .sort_index()
)

yearly_trend

In [ ]:
# Graph 4: Immigration trend over time

plt.figure(figsize=(10, 6))
yearly_trend.plot(kind='line', marker='o')
plt.title('U.S. Immigration Activity Trend, 2013–2023')
plt.xlabel('Year')
plt.ylabel('Total Immigration Activity')
plt.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
# QUESTION 5:
# Which states had the highest immigration activity in 2023?

df_2023 = df_clean[df_clean['Year'] == 2023].copy()

top_2023_states = (
    df_2023.groupby('State')['Total Immigration Activity']
    .sum()
    .sort_values(ascending=False)
    .head(10)
)

top_2023_states

In [ ]:
# Graph 5: Top 2023 states

plt.figure(figsize=(12, 6))
top_2023_states.plot(kind='bar')
plt.title('Top 10 States by Immigration Activity in 2023')
plt.xlabel('State')
plt.ylabel('Total Immigration Activity')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

In [ ]:
# STEP 8: U.S. map visualization

state_abbrev = {
    'Alabama': 'AL', 'Alaska': 'AK', 'Arizona': 'AZ', 'Arkansas': 'AR',
    'California': 'CA', 'Colorado': 'CO', 'Connecticut': 'CT', 'Delaware': 'DE',
    'District of Columbia': 'DC', 'Florida': 'FL', 'Georgia': 'GA', 'Hawaii': 'HI',
    'Idaho': 'ID', 'Illinois': 'IL', 'Indiana': 'IN', 'Iowa': 'IA',
    'Kansas': 'KS', 'Kentucky': 'KY', 'Louisiana': 'LA', 'Maine': 'ME',
    'Maryland': 'MD', 'Massachusetts': 'MA', 'Michigan': 'MI', 'Minnesota': 'MN',
    'Mississippi': 'MS', 'Missouri': 'MO', 'Montana': 'MT', 'Nebraska': 'NE',
    'Nevada': 'NV', 'New Hampshire': 'NH', 'New Jersey': 'NJ', 'New Mexico': 'NM',
    'New York': 'NY', 'North Carolina': 'NC', 'North Dakota': 'ND', 'Ohio': 'OH',
    'Oklahoma': 'OK', 'Oregon': 'OR', 'Pennsylvania': 'PA', 'Rhode Island': 'RI',
    'South Carolina': 'SC', 'South Dakota': 'SD', 'Tennessee': 'TN', 'Texas': 'TX',
    'Utah': 'UT', 'Vermont': 'VT', 'Virginia': 'VA', 'Washington': 'WA',
    'West Virginia': 'WV', 'Wisconsin': 'WI', 'Wyoming': 'WY'
}

state_map_data = (
    df_clean.groupby('State')['Total Immigration Activity']
    .sum()
    .reset_index()
)

state_map_data['State Code'] = state_map_data['State'].map(state_abbrev)

state_map_data.head()

In [ ]:
# U.S. Map Graph

fig = px.choropleth(
    state_map_data.dropna(subset=['State Code']),
    locations='State Code',
    locationmode='USA-states',
    color='Total Immigration Activity',
    hover_name='State',
    scope='usa',
    title='Total Immigration Activity by State, 2013–2023',
    color_continuous_scale='Blues'
)

fig.show()

In [ ]:
# BONUS QUESTION:
# Immigration activity per million residents

state_population = (
    df_clean.sort_values('Year')
    .groupby('State')['Population']
    .last()
)

state_total_activity = df_clean.groupby('State')['Total Immigration Activity'].sum()

per_million = ((state_total_activity / state_population) * 1_000_000).sort_values(ascending=False)

per_million.head(10)

In [ ]:
# Bonus Graph: Immigration activity per million residents

plt.figure(figsize=(12, 6))
per_million.head(10).plot(kind='bar')
plt.title('Top 10 States by Immigration Activity per Million Residents')
plt.xlabel('State')
plt.ylabel('Immigration Activity per Million Residents')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

In [ ]:
class ImmigrationAnalysis:

    def __init__(self, dataframe):
        self.df = dataframe

    def top_states(self):
        return self.df.groupby('State')['Total Immigration Activity'].sum().sort_values(ascending=False).head(10)

    def yearly_trend(self):
        return self.df.groupby('Year')['Total Immigration Activity'].sum()

# Use class
analysis = ImmigrationAnalysis(df_clean)

analysis.top_states()
